In [ ]:
!pip install -q kaggle

!mkdir -p /content/data/fakeddit
!kaggle datasets download -d vanshikavmittal/fakeddit-dataset -p /content/data/fakeddit --unzip

In [ ]:
import polars as pl
from pathlib import Path
import plotly.express as px
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib

SAMPLES_DIR = Path("/content/data/fakeddit/multimodal_only_samples")

all_data: pl.DataFrame = pl.concat([
    pl.read_csv(SAMPLES_DIR / filename, separator="\t") for filename in SAMPLES_DIR.iterdir()
]).with_columns(
    pl.from_epoch("created_utc", time_unit="s").alias("created_date")
)

trimmed_data = all_data.select(
    pl.col("author"),
    pl.col("created_utc"),
    pl.col("clean_title"),
    pl.col("score"),
    pl.col("2_way_label"),
    pl.col("6_way_label"),
    pl.col("num_comments"),
    pl.col("upvote_ratio"),
    pl.col("image_url"),
).unique(
    subset=["clean_title", "image_url"],
    keep="first"
).drop("image_url")
trimmed_data.describe()

In [ ]:
model_data = (
    trimmed_data
    .select(["clean_title", "6_way_label"])
    .drop_nulls()
)

X = model_data["clean_title"]
y = model_data["6_way_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))
print("Train label distribution:")
print(y_train.value_counts(normalize=True))


from sentence_transformers import SentenceTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from typing import Self

class TitleEmbedder(TransformerMixin, BaseEstimator):
    def __init__(self, model_name: str, *, batch_size: int = 256 ) -> None:
        self._model_name = model_name
        self._batch_size = batch_size

    def fit(self, X: np.ndarray, y: None = None) -> Self:
        self._model = SentenceTransformer(self._model_name)
        return self


    def transform(self, X: np.ndarray) -> np.ndarray:
        return self._model.encode(
            list(X),
            batch_size=self._batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
        )

In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path
from tqdm import tqdm
import torch
import gc
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
import lightgbm as lgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression


# -- output dirs for artifacts --
ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)
(ARTIFACTS_DIR / "confusion_matrices").mkdir(exist_ok=True)
(ARTIFACTS_DIR / "errors").mkdir(exist_ok=True)


LABEL_NAMES = [MAP_6_WAY[numeric_label] for numeric_label in range(6)]

In [ ]:

EMBEDDINGS = {
    # "tfidf": ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=30_000, lowercase=True)),
    # "mpnet": ("emb", TitleEmbedder("all-mpnet-base-v2")),
    "minilm-really": ("emb", TitleEmbedder("all-MiniLM-L12-v2")),
    "paraphrase": ("emb", TitleEmbedder("paraphrase-MiniLM-L3-v2")),
}

CLASSIFIERS = {
    "logreg": lambda: LogisticRegression(max_iter=1000, class_weight="balanced"),
    "rf": lambda: RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42,
    ),
    "lgbm": lambda: lgb.LGBMClassifier(
        n_estimators=300,
        learning_rate=0.1,
        class_weight="balanced",
        num_leaves=63,
        n_jobs=-1,
        random_state=42,
        verbose=-1,
    ),
}

MODELS = {}
for emb_name, (step_name, emb_step) in EMBEDDINGS.items():
    for clf_name, clf_factory in CLASSIFIERS.items():
        model_key = f"{emb_name}__{clf_name}"
        MODELS[model_key] = Pipeline([
            (step_name, emb_step),
            ("clf", clf_factory()),
        ])

print(f"Total model configurations: {len(MODELS)}")
for k in MODELS:
    print(f"  • {k}")

In [ ]:

@dataclass
class ModelResult:
    name: str
    roc_auc: float
    f1_macro: float
    accuracy: float
    precision_macro: float
    recall_macro: float


results: list[ModelResult] = []
all_errors: list[pd.DataFrame] = []

for model_name, model in tqdm(MODELS.items()):
    print(f"\n{'='*20} {model_name} {'='*20}")

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)

    roc_auc = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")
    report = classification_report(
        y_test, y_pred, target_names=LABEL_NAMES, digits=4, output_dict=True
    )

    result = ModelResult(
        name=model_name,
        roc_auc=roc_auc,
        f1_macro=report["macro avg"]["f1-score"],
        accuracy=report["accuracy"],
        precision_macro=report["macro avg"]["precision"],
        recall_macro=report["macro avg"]["recall"],
    )
    results.append(result)



    matplotlib.use("Agg")

    cm = confusion_matrix(y_test, y_pred)
    fig_mpl, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cm, cmap="Blues")
    fig_mpl.colorbar(im, ax=ax)

    ax.set_xticks(range(len(LABEL_NAMES)))
    ax.set_yticks(range(len(LABEL_NAMES)))
    ax.set_xticklabels(LABEL_NAMES, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(LABEL_NAMES, fontsize=8)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title(f"{model_name} — ROC-AUC: {roc_auc:.4f} | F1 macro: {result.f1_macro:.4f}")

    # annotate cells with counts
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=7)

    fig_mpl.tight_layout()
    fig_mpl.savefig(ARTIFACTS_DIR / "confusion_matrices" / f"{model_name}.png", dpi=200)
    plt.close(fig_mpl)

    # still show the interactive plotly version in the notebook
    px.imshow(
        cm, x=LABEL_NAMES, y=LABEL_NAMES, text_auto=True,
        color_continuous_scale="Blues",
        labels=dict(x="Predicted label", y="True label", color="Count"),
        title=f"{model_name} — ROC-AUC: {roc_auc:.4f} | F1 macro: {result.f1_macro:.4f}",
        height=600,
    ).show()

    y_test_np = np.array(y_test)
    y_pred_np = np.array(y_pred)
    misclassified_mask = y_test_np != y_pred_np

    if misclassified_mask.any():
        X_test_list = list(X_test)
        error_df = pd.DataFrame({
            "model": model_name,
            "index": np.where(misclassified_mask)[0],
            "text": [X_test_list[i] for i in np.where(misclassified_mask)[0]],
            "true_label": y_test_np[misclassified_mask],
            "true_label_name": [LABEL_NAMES[l] for l in y_test_np[misclassified_mask]],
            "pred_label": y_pred_np[misclassified_mask],
            "pred_label_name": [LABEL_NAMES[l] for l in y_pred_np[misclassified_mask]],
            "pred_proba_max": y_proba[misclassified_mask].max(axis=1),
        })
        error_df.to_csv(
            ARTIFACTS_DIR / "errors" / f"{model_name}_errors.tsv",
            sep="\t", index=False,
        )
        all_errors.append(error_df)

    print(result)

    # free memory between embedding models
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
results_df = pd.DataFrame([asdict(r) for r in results]).sort_values(
    "f1_macro", ascending=False
)
results_df.to_csv(ARTIFACTS_DIR / "results_table.tsv", sep="\t", index=False)
display(results_df)

# -- combined errors file --
if all_errors:
    combined_errors = pd.concat(all_errors, ignore_index=True)
    combined_errors.to_csv(ARTIFACTS_DIR / "all_errors_combined.tsv", sep="\t", index=False)
    print(f"\nTotal misclassifications across all models: {len(combined_errors)}")

print(f"\nAll artifacts saved to: {ARTIFACTS_DIR.resolve()}")